In [16]:
!pip install -q accelerate

In [20]:
%%writefile /kaggle/working/train_ddp_accelerate.py
# pip install accelerate  (su Kaggle: !pip install -q accelerate)
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
import math
import json
import torch
import torch.nn as nn
import torch.optim as optim
import random
import glob
import numpy as np
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
from tqdm import tqdm
import copy

from accelerate import Accelerator
from accelerate.utils import set_seed as accelerate_set_seed

# =========================================================================
# ARCHITETTURA INVARIATA (SelfAttention, UNetBlock, SinusoidalPositionEmbeddings, ImprovedUNet)
# =========================================================================
class SelfAttention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.channels = channels
        self.mha = nn.MultiheadAttention(embed_dim=channels, num_heads=2, batch_first=True)
        self.ln = nn.LayerNorm(channels)
        self.ff_self = nn.Sequential(
            nn.LayerNorm(channels),
            nn.Linear(channels, channels),
            nn.GELU(),
            nn.Linear(channels, channels),
        )

    def forward(self, x):
        size = x.shape[-1]
        x_flat = x.reshape(-1, self.channels, size * size).transpose(1, 2)
        x_ln = self.ln(x_flat)
        attention_value, _ = self.mha(x_ln, x_ln, x_ln)
        attention_value = attention_value + x_flat
        attention_value = self.ff_self(attention_value) + attention_value
        return attention_value.transpose(1, 2).reshape(-1, self.channels, size, size)


class UNetBlock(nn.Module):
    def __init__(self, in_channels, out_channels, time_dim=32, num_groups=4):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.gn1 = nn.GroupNorm(num_groups, out_channels)
        self.act1 = nn.SiLU()

        self.time_mlp = nn.Linear(time_dim, out_channels)

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.gn2 = nn.GroupNorm(num_groups, out_channels)
        self.act2 = nn.SiLU()

        self.residual_conv = nn.Conv2d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else nn.Identity()

    def forward(self, x, t_emb):
        h = self.act1(self.gn1(self.conv1(x)))
        time_proj = self.time_mlp(t_emb).unsqueeze(-1).unsqueeze(-1)
        h = h + time_proj
        h = self.act2(self.gn2(self.conv2(h)))
        return h + self.residual_conv(x)


class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings


class ImprovedUNet(nn.Module):
    def __init__(self, in_channels=3, cond_channels=3, base_channels=64):
        super().__init__()

        time_dim = base_channels * 4
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(base_channels),
            nn.Linear(base_channels, time_dim),
            nn.GELU(),
            nn.Linear(time_dim, time_dim)
        )

        c = base_channels

        self.inc = UNetBlock(in_channels + cond_channels, c, time_dim)

        self.down1 = nn.Conv2d(c, c, kernel_size=4, stride=2, padding=1)
        self.enc1 = UNetBlock(c, c * 2, time_dim)

        self.down2 = nn.Conv2d(c * 2, c * 2, kernel_size=4, stride=2, padding=1)
        self.enc2 = UNetBlock(c * 2, c * 4, time_dim)

        self.down3 = nn.Conv2d(c * 4, c * 4, kernel_size=4, stride=2, padding=1)
        self.enc3 = UNetBlock(c * 4, c * 8, time_dim)
        self.attn3 = SelfAttention(c * 8)

        self.down4 = nn.Conv2d(c * 8, c * 8, kernel_size=4, stride=2, padding=1)

        self.bottleneck1 = UNetBlock(c * 8, c * 8, time_dim)
        self.attention = SelfAttention(c * 8)
        self.bottleneck2 = UNetBlock(c * 8, c * 8, time_dim)

        self.up1 = nn.ConvTranspose2d(c * 8, c * 8, kernel_size=4, stride=2, padding=1)
        self.dec1 = UNetBlock(c * 16, c * 4, time_dim)
        self.attn_up1 = SelfAttention(c * 4)

        self.up2 = nn.ConvTranspose2d(c * 4, c * 4, kernel_size=4, stride=2, padding=1)
        self.dec2 = UNetBlock(c * 8, c * 2, time_dim)

        self.up3 = nn.ConvTranspose2d(c * 2, c * 2, kernel_size=4, stride=2, padding=1)
        self.dec3 = UNetBlock(c * 4, c, time_dim)

        self.up4 = nn.ConvTranspose2d(c, c, kernel_size=4, stride=2, padding=1)
        self.dec4 = UNetBlock(c * 2, c, time_dim)

        self.out = nn.Conv2d(c, in_channels, kernel_size=3, padding=1)

    def forward(self, x_t, t, condition):
        x_input = torch.cat([x_t, condition], dim=1)
        t_emb = self.time_mlp(t.float())

        s1 = self.inc(x_input, t_emb)
        h = self.down1(s1)

        s2 = self.enc1(h, t_emb)
        h = self.down2(s2)

        s3 = self.enc2(h, t_emb)
        h = self.down3(s3)

        s4 = self.enc3(h, t_emb)
        s4 = self.attn3(s4)
        h = self.down4(s4)

        h = self.bottleneck1(h, t_emb)
        h = self.attention(h)
        h = self.bottleneck2(h, t_emb)

        h = self.up1(h)
        h = torch.cat([h, s4], dim=1)
        h = self.dec1(h, t_emb)
        h = self.attn_up1(h)

        h = self.up2(h)
        h = torch.cat([h, s3], dim=1)
        h = self.dec2(h, t_emb)

        h = self.up3(h)
        h = torch.cat([h, s2], dim=1)
        h = self.dec3(h, t_emb)

        h = self.up4(h)
        h = torch.cat([h, s1], dim=1)
        h = self.dec4(h, t_emb)

        return self.out(h)


# =========================================================================
def cosine_beta_schedule(timesteps, s=0.008):
    steps = timesteps + 1
    x = torch.linspace(0, timesteps, steps)
    alphas_cumprod = torch.cos(((x / timesteps) + s) / (1 + s) * math.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clip(betas, 1e-4, 0.9999)


class DDPMvPrediction(nn.Module):
    def __init__(self, network, n_steps=300, beta_start=1e-4, beta_end=0.02):
        super().__init__()
        self.network = network
        self.n_steps = n_steps

        beta = cosine_beta_schedule(n_steps)
        alpha = 1.0 - beta
        alpha_bar = torch.cumprod(alpha, dim=0)
        alpha_bar_prev = torch.cat([torch.tensor([1.0]), alpha_bar[:-1]])

        sqrt_alpha_bar = torch.sqrt(alpha_bar)
        sqrt_one_minus_alpha_bar = torch.sqrt(1.0 - alpha_bar)

        self.register_buffer('alpha_bar', alpha_bar)
        self.register_buffer('alpha_bar_prev', alpha_bar_prev)
        self.register_buffer('sqrt_alpha_bar', sqrt_alpha_bar)
        self.register_buffer('sqrt_one_minus_alpha_bar', sqrt_one_minus_alpha_bar)
        self.register_buffer('beta', beta)
        self.register_buffer('alpha', alpha)
        self.register_buffer('posterior_variance', beta * (1 - alpha_bar_prev) / (1 - alpha_bar))

    def forward_diffusion(self, x_0, t, noise):
        sqrt_alpha_bar_t = self.sqrt_alpha_bar[t].view(-1, 1, 1, 1)
        sqrt_one_minus_alpha_bar_t = self.sqrt_one_minus_alpha_bar[t].view(-1, 1, 1, 1)
        return sqrt_alpha_bar_t * x_0 + sqrt_one_minus_alpha_bar_t * noise

    def forward(self, x_0, condition):
        batch_size = x_0.shape[0]
        t = torch.randint(0, self.n_steps, (batch_size,), device=x_0.device)
        noise = torch.randn_like(x_0)

        x_t = self.forward_diffusion(x_0, t, noise)

        sqrt_alpha_bar_t = self.sqrt_alpha_bar[t].view(-1, 1, 1, 1)
        sqrt_one_minus_alpha_bar_t = self.sqrt_one_minus_alpha_bar[t].view(-1, 1, 1, 1)

        v_target = sqrt_alpha_bar_t * noise - sqrt_one_minus_alpha_bar_t * x_0
        predicted_v = self.network(x_t, t.float(), condition)

        loss = F.mse_loss(predicted_v, v_target)
        return loss

    @torch.no_grad()
    def sample(self, condition):
        self.network.eval()
        device = self.beta.device

        batch_size, channels, height, width = condition.shape
        x = torch.randn(batch_size, channels, height, width, device=device)

        for i in reversed(range(self.n_steps)):
            t = torch.full((batch_size,), i, dtype=torch.long, device=device)
            predicted_v = self.network(x, t.float(), condition)

            sqrt_alpha_bar_t = self.sqrt_alpha_bar[i]
            sqrt_one_minus_alpha_bar_t = self.sqrt_one_minus_alpha_bar[i]

            pred_x0 = (sqrt_alpha_bar_t * x) - (sqrt_one_minus_alpha_bar_t * predicted_v)
            pred_x0 = torch.clamp(pred_x0, -1.0, 1.0)

            beta_t = self.beta[i]
            alpha_bar_prev_t = self.alpha_bar_prev[i]
            alpha_bar_t = self.alpha_bar[i]
            alpha_t = self.alpha[i]

            mean = (beta_t * torch.sqrt(alpha_bar_prev_t) / (1.0 - alpha_bar_t)) * pred_x0 + \
                   ((1.0 - alpha_bar_prev_t) * torch.sqrt(alpha_t) / (1.0 - alpha_bar_t)) * x

            if i > 0:
                noise = torch.randn_like(x)
                sigma_t = torch.sqrt(self.posterior_variance[i])
                x = mean + sigma_t * noise
            else:
                x = mean

        self.network.train()
        return x


def strip_compile_prefix(state_dict):
    if not state_dict:
        return state_dict
    if all(k.startswith("_orig_mod.") for k in state_dict.keys()):
        return {k[len("_orig_mod."):]: v for k, v in state_dict.items()}
    return state_dict


class ExponentialMovingAverage:
    def __init__(self, beta=0.999):
        self.beta = beta

    def update(self, ema_model, current_model):
        with torch.no_grad():
            for ema_param, current_param in zip(ema_model.parameters(), current_model.parameters()):
                ema_param.data.mul_(self.beta).add_(current_param.data, alpha=1.0 - self.beta)


class StarRemovalDataset(Dataset):
    """
    Puo' essere costruito in due modi:
    1) a partire da due directory (starry_dir, starless_dir): elenca e ordina i file .npy
       al loro interno (comportamento originale).
    2) a partire da due liste esplicite di path (starry_paths, starless_paths): usato per
       ricostruire ESATTAMENTE lo stesso split val/extra-train salvato nel manifest, senza
       dipendere da sorted(os.listdir()) che puo' cambiare se il contenuto delle cartelle cambia.
    """
    def __init__(self, starry_dir=None, starless_dir=None, starry_paths=None, starless_paths=None):
        if starry_paths is not None and starless_paths is not None:
            self.starry_paths = list(starry_paths)
            self.starless_paths = list(starless_paths)
        else:
            self.starry_paths = sorted([os.path.join(starry_dir, f) for f in os.listdir(starry_dir) if f.endswith('.npy')])
            self.starless_paths = sorted([os.path.join(starless_dir, f) for f in os.listdir(starless_dir) if f.endswith('.npy')])

        assert len(self.starry_paths) == len(self.starless_paths)
        assert len(self.starry_paths) > 0

    def __len__(self):
        return len(self.starry_paths)

    def __getitem__(self, idx):
        starry_arr = np.load(self.starry_paths[idx])
        starless_arr = np.load(self.starless_paths[idx])

        starry_tensor = torch.from_numpy(starry_arr).float()
        starless_tensor = torch.from_numpy(starless_arr).float()

        if starry_tensor.ndim == 3 and starry_tensor.shape[-1] in [1, 3]:
            starry_tensor = starry_tensor.permute(2, 0, 1)
            starless_tensor = starless_tensor.permute(2, 0, 1)

        starry_tensor = (starry_tensor * 2.0) - 1.0
        starless_tensor = (starless_tensor * 2.0) - 1.0

        return starry_tensor, starless_tensor


# =========================================================================
# TRAINING (DDP tramite accelerate)
# =========================================================================
USE_TORCH_COMPILE = False

def main():
    accelerator = Accelerator(mixed_precision="fp16")
    accelerate_set_seed(42)
    device = accelerator.device

    torch.backends.cudnn.benchmark = True

    output_dir = "/kaggle/working/Risultati_DDPM_vpred"
    if accelerator.is_main_process:
        os.makedirs(output_dir, exist_ok=True)
    accelerator.wait_for_everyone()

    input_checkpoint_dir = "/kaggle/input/datasets/ilariadarchivio"

    if accelerator.is_main_process:
        if os.path.exists(input_checkpoint_dir):
            print(f"input_checkpoint_dir trovato: {input_checkpoint_dir} -> contenuto: {os.listdir(input_checkpoint_dir)}")
        else:
            print(f"ATTENZIONE: input_checkpoint_dir non esiste in questa sessione: {input_checkpoint_dir}")

    train_starry_dir = "/kaggle/input/datasets/phantasm04/star-removal-uar/dataset/input/npy"
    train_starless_dir = "/kaggle/input/datasets/phantasm04/star-removal-uar/dataset/target/npy"
    
    val_starry_dir = "/kaggle/input/datasets/phantasm04/star-removal/dataset/input/npy"
    val_starless_dir = "/kaggle/input/datasets/phantasm04/star-removal/dataset/target/npy"
    
    # NOTA: non carichiamo piu' train_dataset (cartella uar intera) in modo
    # incondizionato. Se esiste un manifest, e' il manifest a decidere quali
    # campioni uar entrano nel training: caricare anche la cartella intera
    # duplicherebbe quei campioni.
    full_val_dataset = StarRemovalDataset(val_starry_dir, val_starless_dir)
    
    # =====================================================================
    # SPLIT VAL/TRAIN:
    # Se esiste gia' un manifest (sessione precedente, o generato da un altro
    # processo di split che decide anche la quota di uar da usare) lo
    # ricarichiamo per garantire lo stesso identico split, invece di
    # ri-derivarlo da sorted(os.listdir()) + seed.
    # =====================================================================
    split_manifest_path_out = os.path.join(output_dir, "split_manifest.json")
    split_manifest_path_in = os.path.join(input_checkpoint_dir, "split-dataset", "split_manifest.json")
    
    existing_manifest_path = None
    if os.path.exists(split_manifest_path_out):
        existing_manifest_path = split_manifest_path_out
    elif os.path.exists(split_manifest_path_in):
        existing_manifest_path = split_manifest_path_in
    
    if existing_manifest_path is not None:
        if accelerator.is_main_process:
            print(f"Trovato manifest di split esistente in: {existing_manifest_path}. Lo ricarico per coerenza.")
        with open(existing_manifest_path, "r") as f:
            split_manifest = json.load(f)
    
        required_keys = {"val_input", "val_target", "train_input", "train_target"}
        missing_keys = required_keys - split_manifest.keys()
        if missing_keys:
            raise KeyError(f"Il manifest {existing_manifest_path} non contiene le chiavi attese: {missing_keys}")
    
        val_dataset = StarRemovalDataset(
            starry_paths=split_manifest["val_input"],
            starless_paths=split_manifest["val_target"]
        )
        # IMPORTANTE: quando esiste un manifest, "train_input"/"train_target" sono
        # il training set GIA' COMPLETO (include la quota di uar + la quota di
        # normal decise da chi ha generato il manifest). NON concateniamo piu'
        # la cartella uar intera sopra a questo: altrimenti i campioni uar gia'
        # presenti nel manifest verrebbero duplicati.
        combined_train_dataset = StarRemovalDataset(
            starry_paths=split_manifest["train_input"],
            starless_paths=split_manifest["train_target"]
        )
    
        val_subset_size = len(val_dataset)
        train_total_size = len(combined_train_dataset)
    
        # --- CHECK DI COERENZA A RUNTIME ---
        if accelerator.is_main_process:
            manifest_val_paths = set(split_manifest["val_input"])
            manifest_train_paths = set(split_manifest["train_input"])
    
            overlap_val_train = manifest_val_paths & manifest_train_paths
            if overlap_val_train:
                print(f"⚠️ ATTENZIONE: {len(overlap_val_train)} campioni compaiono sia in val_input che in train_input nel manifest!")
    
            uar_folder_files = set(
                os.path.join(train_starry_dir, f) for f in os.listdir(train_starry_dir) if f.endswith(".npy")
            )
            uar_train_in_manifest = manifest_train_paths & uar_folder_files
            uar_val_in_manifest = manifest_val_paths & uar_folder_files
    
            normal_folder_files = set(full_val_dataset.starry_paths)
            normal_train_in_manifest = manifest_train_paths & normal_folder_files
            normal_val_in_manifest = manifest_val_paths & normal_folder_files
    
            unaccounted_train = manifest_train_paths - uar_folder_files - normal_folder_files
            unaccounted_val = manifest_val_paths - uar_folder_files - normal_folder_files
    
            print(f"Manifest caricato: {val_subset_size} campioni di validazione, {train_total_size} campioni di training (gia' combinati, uar+normal).")
            print(f"  Training  -> da uar: {len(uar_train_in_manifest)} | da normal: {len(normal_train_in_manifest)} | non riconosciuti: {len(unaccounted_train)}")
            print(f"  Validation -> da uar: {len(uar_val_in_manifest)} | da normal: {len(normal_val_in_manifest)} | non riconosciuti: {len(unaccounted_val)}")
            if unaccounted_train or unaccounted_val:
                print("  ⚠️ ATTENZIONE: alcuni path del manifest non corrispondono a file presenti nelle cartelle attuali "
                      "(dataset cambiato rispetto alla creazione del manifest, o path assoluti diversi tra sessioni).")
    
    else:
        val_total_size = len(full_val_dataset)
        val_subset_size = int(0.15 * val_total_size)
        extra_train_size = val_total_size - val_subset_size  # I dati precedentemente scartati
    
        # Dividiamo il dataset star-removal: 15% per la validazione, 85% da unire al training
        val_dataset, extra_train_dataset = torch.utils.data.random_split(
            full_val_dataset, [val_subset_size, extra_train_size],
            generator=torch.Generator().manual_seed(42))
    
        split_manifest = {
            "val_input": [full_val_dataset.starry_paths[i] for i in val_dataset.indices],
            "val_target": [full_val_dataset.starless_paths[i] for i in val_dataset.indices],
            "train_input": [full_val_dataset.starry_paths[i] for i in extra_train_dataset.indices],
            "train_target": [full_val_dataset.starless_paths[i] for i in extra_train_dataset.indices],
        }
        if accelerator.is_main_process:
            with open(split_manifest_path_out, "w") as f:
                json.dump(split_manifest, f, indent=2)
            print(f"Manifest di split creato e salvato in: {split_manifest_path_out}")
    
        # Nessun manifest esterno: comportamento originale, uar intera + quota extra da normal
        train_dataset = StarRemovalDataset(train_starry_dir, train_starless_dir)
        combined_train_dataset = torch.utils.data.ConcatDataset([train_dataset, extra_train_dataset])
        train_total_size = len(combined_train_dataset)
    
        if accelerator.is_main_process:
            print(f"Dataset di training base ('star-removal-uar'): {len(train_dataset)} campioni.")
            print(f"Dataset 'star-removal': {val_total_size} campioni -> {val_subset_size} per validazione, {extra_train_size} recuperati per il training.")
    
    if accelerator.is_main_process:
        print(f"Dataset di training TOTALE: {train_total_size} campioni.")
        print(f"Dataset di validazione TOTALE: {val_subset_size} campioni.")
    accelerator.wait_for_everyone()
    
    train_dataloader = DataLoader(combined_train_dataset, batch_size=4, shuffle=True, drop_last=True,
                                   num_workers=4, pin_memory=True, persistent_workers=True)
    val_dataloader = DataLoader(val_dataset, batch_size=4, shuffle=False, drop_last=False,
                                 num_workers=4, pin_memory=True, persistent_workers=True)

    epochs = 1000
    max_epoch_this_session = 100

    nn_architecture = ImprovedUNet(in_channels=3, base_channels=64)

    # =====================================================================
    # SALVATAGGIO PESI INIZIALI (theta0): dal checkpoint di input caricato.
    # =====================================================================
    theta0_path_out = os.path.join(output_dir, "theta0.pth")
    theta0_path_in = os.path.join(input_checkpoint_dir, "theta0-pretrain", "theta0.pth")

    # 1. Controllo di sicurezza su tutti i processi per evitare blocchi (deadlock)
    if not os.path.exists(theta0_path_out) and not os.path.exists(theta0_path_in):
        raise FileNotFoundError(
            f"[ERRORE] theta0 non è stato trovato."
            f"Il processo è stato interrotto come richiesto."
        )
    
    # 2. Se i file esistono, il processo principale gestisce i log di conferma
    if accelerator.is_main_process:
        if os.path.exists(theta0_path_in):
            print(f"theta0 presente nel checkpoint di input ({theta0_path_in}), pronto per l'uso.")
        else:
            print(f"theta0 presente in output_dir ({theta0_path_out}), pronto per l'uso.")
    
    # 3. Sincronizzazione finale dei processi
    accelerator.wait_for_everyone()

    ema_architecture = copy.deepcopy(nn_architecture).to(device)
    ema_architecture.eval()
    ema_updater = ExponentialMovingAverage(beta=0.999)

    ddpm = DDPMvPrediction(nn_architecture, n_steps=200)
    ddpm_ema = DDPMvPrediction(ema_architecture, n_steps=200).to(device)
    ddpm_ema.eval()

    optimizer = optim.AdamW(ddpm.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    ddpm, optimizer, train_dataloader, val_dataloader, scheduler = accelerator.prepare(
        ddpm, optimizer, train_dataloader, val_dataloader, scheduler
    )

    if USE_TORCH_COMPILE:
        ddpm = torch.compile(ddpm)

    start_epoch = 0
    history_train_loss = []
    history_val_loss = []

    # --- NUOVE VARIABILI PER TRACCIARE IL MIGLIOR MODELLO ---
    best_val_loss = float('inf')
    best_epoch = -1

    checkpoints_out = glob.glob(os.path.join(output_dir, "ddpmvpred_epoch_*.pth"))
    checkpoints_in = []
    if os.path.exists(input_checkpoint_dir):
        checkpoints_in = glob.glob(os.path.join(input_checkpoint_dir, "ddpmvpred_epoch_*.pth"))
    all_checkpoints = checkpoints_out + checkpoints_in

    if all_checkpoints:
        all_checkpoints.sort(key=lambda x: int(os.path.basename(x).split('_')[-1].split('.')[0]))
        last_checkpoint = all_checkpoints[-1]

        if accelerator.is_main_process:
            print(f"Trovato checkpoint precedente in: {last_checkpoint}")
        checkpoint = torch.load(last_checkpoint, map_location=device)

        unwrapped_ddpm = accelerator.unwrap_model(ddpm)
        unwrapped_ddpm.load_state_dict(strip_compile_prefix(checkpoint["model"]))
        optimizer.load_state_dict(checkpoint["optimizer"])
        scheduler.load_state_dict(checkpoint["scheduler"])
        start_epoch = checkpoint["epoch"] + 1

        if "ema_model" in checkpoint:
            ema_architecture.load_state_dict(strip_compile_prefix(checkpoint["ema_model"]))
        else:
            if accelerator.is_main_process:
                print("Pesi EMA non trovati, inizializzo EMA dai pesi attuali.")
            ema_architecture.load_state_dict(unwrapped_ddpm.network.state_dict())

        if "history_train_loss" in checkpoint:
            history_train_loss = checkpoint["history_train_loss"]
            history_val_loss = checkpoint.get("history_val_loss", [0] * len(history_train_loss))

        # --- CARICAMENTO VALORI BEST PRECEDENTI ---
        if "best_val_loss" in checkpoint:
            best_val_loss = checkpoint["best_val_loss"]
            best_epoch = checkpoint.get("best_epoch", -1)
            if accelerator.is_main_process:
                print(f"Ripristinata miglior validation loss precedente: {best_val_loss:.4f} (Epoca {best_epoch})")
    else:
        # Nessun checkpoint di training (ddpmvpred_epoch_*.pth) trovato: prima di
        # inizializzare random, controlliamo se esiste un theta0 da usare come
        # punto di partenza comune, coerente tra le varie sessioni.
        theta0_to_load = None
        if os.path.exists(theta0_path_out):
            theta0_to_load = theta0_path_out
        elif os.path.exists(theta0_path_in):
            theta0_to_load = theta0_path_in

        if theta0_to_load is not None:
            if accelerator.is_main_process:
                print(f"Nessun checkpoint di training trovato. Carico theta0 come punto di partenza: {theta0_to_load}")
            theta0_state = strip_compile_prefix(torch.load(theta0_to_load, map_location=device))
            unwrapped_ddpm = accelerator.unwrap_model(ddpm)
            unwrapped_ddpm.network.load_state_dict(theta0_state)
            ema_architecture.load_state_dict(theta0_state)
        else:
            if accelerator.is_main_process:
                print("Inizio l'addestramento v-prediction da zero (nessun theta0 trovato).")

    fixed_indices = [0, 16, 24, 34]
    val_condition_list, val_target_list = [], []
    for idx in fixed_indices:
        safe_idx = idx if idx < len(val_dataset) else (idx % len(val_dataset))
        cond, target = val_dataset[safe_idx]
        val_condition_list.append(cond)
        val_target_list.append(target)

    fixed_val_condition_cpu = torch.stack(val_condition_list)
    fixed_val_target_cpu = torch.stack(val_target_list)
    if accelerator.is_main_process:
        print(f"Batch di validazione fisso preparato con gli indici: {fixed_indices}")
        print(f"Inizio del training v-prediction ({epochs} epoche) su {accelerator.num_processes} GPU...")

    for epoch in range(start_epoch, epochs):
        # -- 1. TRAINING --
        ddpm.train()
        total_train_loss = 0.0
        pbar = tqdm(train_dataloader, desc=f"Epoch {epoch+1:03d}/{epochs} [Train]",
                    disable=not accelerator.is_local_main_process)
        for batch in pbar:
            condition, x_0 = batch

            optimizer.zero_grad(set_to_none=True)
            loss = ddpm(x_0, condition)

            accelerator.backward(loss)
            accelerator.clip_grad_norm_(ddpm.parameters(), max_norm=1.0)
            optimizer.step()

            ema_updater.update(ema_architecture, accelerator.unwrap_model(ddpm).network)

            total_train_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        avg_train_loss = total_train_loss / len(train_dataloader)
        history_train_loss.append(avg_train_loss)

        # -- 2. VALIDAZIONE (con modello EMA) --
        ddpm_ema.eval()
        total_val_loss = 0.0
        with torch.no_grad():
            for batch in val_dataloader:
                condition, x_0 = batch
                val_loss = ddpm_ema(x_0, condition)
                total_val_loss += val_loss.item()

        avg_val_loss = total_val_loss / len(val_dataloader)
        history_val_loss.append(avg_val_loss)

        if accelerator.is_main_process:
            print(f"Epoch {epoch+1:03d} | Train Loss: {avg_train_loss:.4f} | Val Loss (EMA): {avg_val_loss:.4f}")

            # --- 2.5 SALVATAGGIO BEST MODEL ---
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                best_epoch = epoch + 1
                print(f"🌟 Nuova miglior validation loss ({best_val_loss:.4f})! Salvataggio del modello best in corso...")

                unwrapped_ddpm = accelerator.unwrap_model(ddpm)
                best_checkpoint = {
                    "epoch": epoch,
                    "model": unwrapped_ddpm.state_dict(),
                    "ema_model": ema_architecture.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict(),
                    "history_train_loss": history_train_loss,
                    "history_val_loss": history_val_loss,
                    "best_val_loss": best_val_loss,
                    "best_epoch": best_epoch
                }
                torch.save(best_checkpoint, os.path.join(output_dir, "ddpmvpred_best.pth"))

        scheduler.step()
        accelerator.wait_for_everyone()

        # -- 3. SALVATAGGIO REGOLARE E VISUALIZZAZIONE --
        if (epoch + 1) % 5 == 0 and accelerator.is_main_process:
            print(f"\n---> Generazione test v-prediction all'epoca {epoch+1} (EMA)...")

            unwrapped_ddpm = accelerator.unwrap_model(ddpm)
            torch.save({
                "epoch": epoch,
                "model": unwrapped_ddpm.state_dict(),
                "ema_model": ema_architecture.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "history_train_loss": history_train_loss,
                "history_val_loss": history_val_loss,
                "best_val_loss": best_val_loss, # AGGIUNTO NEL SALVATAGGIO REGOLARE
                "best_epoch": best_epoch        # AGGIUNTO NEL SALVATAGGIO REGOLARE
            }, os.path.join(output_dir, f"ddpmvpred_epoch_{epoch+1}.pth"))

            KEEP_LAST_N_CHECKPOINTS = 2
            saved_this_session = sorted(
                glob.glob(os.path.join(output_dir, "ddpmvpred_epoch_*.pth")),
                key=lambda p: int(os.path.basename(p).split('_')[-1].split('.')[0])
            )
            for old_ckpt in saved_this_session[:-KEEP_LAST_N_CHECKPOINTS]:
                try:
                    os.remove(old_ckpt)
                    print(f"Rimosso vecchio checkpoint per liberare spazio: {os.path.basename(old_ckpt)}")
                except OSError as e:
                    print(f"Impossibile rimuovere {old_ckpt}: {e}")

            plt.figure(figsize=(10, 5))
            epoche_range = range(1, len(history_train_loss) + 1)
            plt.plot(epoche_range, history_train_loss, label='Train Loss', color='blue')
            plt.plot(epoche_range, history_val_loss, label='Val Loss (EMA)', color='orange')

            # --- AGGIUNTO UN MARKER PER VISUALIZZARE L'EPOCA DEL BEST MODEL SUL GRAFICO ---
            if best_epoch > 0:
                plt.axvline(x=best_epoch, color='red', linestyle='--', alpha=0.5, label=f'Best Epoch ({best_epoch})')

            plt.yscale('log')
            plt.title(f"Andamento Loss (Log Scale) - Epoca {epoch+1}")
            plt.xlabel("Epoche")
            plt.ylabel("MSE Loss (v-prediction)")
            plt.legend()
            plt.grid(True, which="both", ls="--", alpha=0.5)
            plt.tight_layout()
            plt.savefig(os.path.join(output_dir, f"loss_epoch_{epoch+1}.png"))
            plt.close()

            torch.cuda.empty_cache()

            with torch.no_grad():
                val_batch = fixed_val_target_cpu.to(device)
                val_condition = fixed_val_condition_cpu.to(device)
                generated_samples = ddpm_ema.sample(condition=val_condition)

            cond_vis = torch.clamp((val_condition.cpu() + 1.0) / 2.0, 0.0, 1.0)
            gen_vis = torch.clamp((generated_samples.cpu() + 1.0) / 2.0, 0.0, 1.0)
            real_vis = torch.clamp((val_batch.cpu() + 1.0) / 2.0, 0.0, 1.0)

            grid = make_grid(torch.cat([cond_vis, gen_vis, real_vis], dim=0),
                              nrow=fixed_val_condition_cpu.shape[0]).permute(1, 2, 0).numpy()

            plt.figure(figsize=(12, 9))
            plt.imshow(grid)
            plt.title(f"Epoca {epoch+1} v-pred (Indici {fixed_indices}) - EMA\n"
                      f"In alto: Input -> In mezzo: Predizione -> In basso: Target")
            plt.axis('off')
            plt.savefig(os.path.join(output_dir, f"samples_epoch_{epoch+1}.png"))
            plt.close()
            torch.cuda.empty_cache()

        accelerator.wait_for_everyone()

        # -- 4. LIMITE DI SESSIONE --
        if (epoch + 1) >= max_epoch_this_session:
            if accelerator.is_main_process:
                print(f"\nLimite epoche per questa sessione raggiunto ({max_epoch_this_session}). Stop pulito.")
            break

    if accelerator.is_main_process:
        print("\nTraining Completato con Successo!")


if __name__ == "__main__":
    main()

Overwriting /kaggle/working/train_ddp_accelerate.py


In [21]:
!accelerate launch --multi_gpu --num_processes=2 --mixed_precision=fp16 /kaggle/working/train_ddp_accelerate.py

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
input_checkpoint_dir trovato: /kaggle/input/datasets/ilariadarchivio -> contenuto: ['split-dataset', 'theta0-pretrain']
Trovato manifest di split esistente in: /kaggle/input/datasets/ilariadarchivio/split-dataset/split_manifest.json. Lo ricarico per coerenza.
Manifest caricato: 1680 campioni di validazione, 1920 campioni di training (gia' combinati, uar+normal).
  Training  -> da uar: 1200 | da normal: 720 | non riconosciuti: 0
  Validation -> da uar: 0 | da normal: 1680 | non riconosciuti: 0
Dataset di training TOTALE: 1920 campioni.
Dataset di validazione TOTALE: 1680 campioni.
theta0 presente nel checkpoint di input (/kaggle/input/datasets/ilariadarchivio/theta0-pretrain/theta0.pth), pronto per l'uso